In [2]:
import pandas as pd
from transformers import pipeline, AutoTokenizer
import re

/Users/emeliejohnsson/RoBERTa-application-success-classification/distilbert_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load the main dataset and supporting files
data = pd.read_csv("raw_data/dataset.csv")
common_names = pd.read_csv("utils/common_names.csv")
gender_neutral_job_titles = pd.read_csv("raw_data/gender_neutral_job_titles.csv")
data[["Resume"]].to_csv("raw_data/resumes.csv", index=False)

In [4]:
# Save smaller subsets for faster testing
pd.DataFrame({"Resume":[data["Resume"][0]]}).to_csv("raw_data/first_resume.csv", index=False)
pd.DataFrame({"Resume": data["Resume"][:3]}).to_csv("raw_data/f0_t3_resumes.csv", index=False)
pd.DataFrame({"Resume": data["Resume"][:100]}).to_csv("raw_data/f0_t100_resumes.csv", index=False)
pd.DataFrame({"Resume": data["Resume"][:500]}).to_csv("raw_data/f0_t500_resumes.csv", index=False)
pd.DataFrame({"Resume": data["Resume"][1000:1500]}).to_csv("raw_data/f1000_t1500_resumes.csv", index=False)

In [5]:
# Words that reveal gender, used to detect and replace gendered language in resumes
gender_words = {
    'female': ['woman', 'girl', 'lady', 'mother', 'wife', 'daughter', 'sister', 'mom'],
    'male': ['man', 'boy', 'boys', 'gentleman', 'father', 'husband', 'son', 'brother', 'dad']
}

In [6]:
ner_pipeline = pipeline("ner",
                        model="ab-ai/pii_model_based_on_distilbert",
                        aggregation_strategy="simple",
                        device=-1)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 2941.77it/s]

In [7]:
tokenizer = AutoTokenizer.from_pretrained("ab-ai/pii_model_based_on_distilbert")

def predict_entities_chunked(text, chunk_size=400, overlap=50):
    """Run NER on long text by splitting it into overlapping token chunks.
    Offsets are adjusted so all entities refer to positions in the original text."""
    encoding = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    input_ids = encoding['input_ids']
    offsets = encoding['offset_mapping']

    all_entities = []
    start = 0

    while start < len(input_ids):
        end = min(start + chunk_size, len(input_ids))
        chunk_ids = input_ids[start:end]
        chunk_text = tokenizer.decode(chunk_ids)
        char_offset = offsets[start][0]  # where this chunk starts in the original text

        results = ner_pipeline(chunk_text)
        for entity in results:
            # Shift offsets from chunk position to full text position
            entity['start'] += char_offset
            entity['end'] += char_offset
            all_entities.append(entity)

        # Advance with overlap to avoid missing entities that span chunk boundaries
        start += chunk_size - overlap

    return all_entities

In [8]:
results = predict_entities_chunked(data["Resume"][0])
for entity in results:
    print(entity)

{'entity_group': 'FIRSTNAME', 'score': np.float32(0.86959046), 'word': 'jason', 'start': 33, 'end': 38}
{'entity_group': 'LASTNAME', 'score': np.float32(0.858773), 'word': 'jones :', 'start': 39, 'end': 46}
{'entity_group': 'FIRSTNAME', 'score': np.float32(0.99649066), 'word': 'jason', 'start': 47, 'end': 52}
{'entity_group': 'LASTNAME', 'score': np.float32(0.99204445), 'word': 'jones', 'start': 53, 'end': 58}
{'entity_group': 'PHONENUMBER', 'score': np.float32(0.979824), 'word': '555 - 123 - 4567 *', 'start': 190, 'end': 208}
{'entity_group': 'DATE', 'score': np.float32(0.90616983), 'word': '- 2018 ) *', 'start': 1433, 'end': 1443}


### Functions for anonymization

In [9]:
def indentified_name(index, extracted_info_dict):
    """Return True if a name was already found for this resume."""
    if not extracted_info_dict[index].get('name'):
        return False
    return True

In [10]:
def anonymize_names(text, index, extracted_info_dict):
    """Find person names using NER and replace them with [PERSON].
    Also saves the full name to the info dictionary. (older version)"""
    if not isinstance(text, str):
        return text

    if index not in extracted_info_dict:
        extracted_info_dict[index] = {'name': []}

    entities = predict_entities_chunked(text)  # was: ner_pipeline(text)
    entities = sorted(entities, key=lambda x: x['start'])

    # Collect name parts keyed by their position in the text
    name_parts = {}
    for entity in entities:
        if entity['entity_group'] in ('FIRSTNAME', 'LASTNAME', 'MIDDLENAME'):
            name_parts[entity['start']] = entity['word']

    if name_parts:
        full_name = ' '.join(name_parts[k] for k in sorted(name_parts))
        extracted_info_dict[index]['name'].append(full_name)

    # Replace name tokens in reverse order to preserve correct character positions
    for entity in sorted(entities, key=lambda x: x['start'], reverse=True):
        if entity['entity_group'] in ('FIRSTNAME', 'LASTNAME', 'MIDDLENAME'):
            text = text[:entity['start']] + '[PERSON]' + text[entity['end']:]

    return text

In [11]:
def anonymize_names(text, index, extracted_info_dict):
    """Find person names using NER and replace them with [PERSON].
    Only records the first occurrence of each name type to avoid duplicates."""
    if not isinstance(text, str):
        return text

    if index not in extracted_info_dict:
        extracted_info_dict[index] = {'name': []}

    entities = predict_entities_chunked(text)
    entities = sorted(entities, key=lambda x: x['start'])

    # Keep only the first occurrence of FIRSTNAME and LASTNAME
    name_parts = {}
    for entity in entities:
        if entity['entity_group'] in ('FIRSTNAME', 'LASTNAME'):
            if entity['entity_group'] not in name_parts:
                name_parts[entity['entity_group']] = entity['word']

    if name_parts and not extracted_info_dict[index]['name']:
        order = ['FIRSTNAME', 'MIDDLENAME', 'LASTNAME']
        full_name = ' '.join(name_parts[k] for k in order if k in name_parts)
        # Strip trailing punctuation that the model sometimes attaches to the word
        full_name = re.sub(r'[\s:,.\-_|]+$', '', full_name).strip()

        if full_name:
            extracted_info_dict[index]['name'].append(full_name)

        # Clean up any trailing punctuation from all saved names
        extracted_info_dict[index]['name'] = [
            re.sub(r'[\s:,.\-_|]+$', '', name) for name in extracted_info_dict[index]['name']]

    # Replace in reverse order to preserve correct character positions
    for entity in sorted(entities, key=lambda x: x['start'], reverse=True):
        if entity['entity_group'] in ('FIRSTNAME', 'LASTNAME'):
            if text[entity['start']:entity['end']].startswith('['):
                continue  # skip if already replaced
            text = text[:entity['start']] + '[PERSON]' + text[entity['end']:]

    return text

----------------------------------------------------
#### Identify all emails

In [12]:
def anonymize_emails(text, index, extracted_info_dict):
    """Find email addresses and replace them with [EMAIL].
    Only runs if a name was already identified, since emails often contain the person's name."""
    if not isinstance(text, str):
        return text
    
    email_pattern = r'[\w\.-]+@[\w\.-]+'
    front_email_pattern = r'[\w\.-]+@'
    emails = re.findall(email_pattern, text)

    if not indentified_name(index, extracted_info_dict):
        return text

    if index not in extracted_info_dict:
        extracted_info_dict[index] = {
            'emails': [],
        }
    
    if emails:
        extracted_info_dict[index]['emails'] = []
        for email in emails:
            if email not in extracted_info_dict[index]['emails']:
                extracted_info_dict[index]['emails'] = email
        # Use negative lookahead and lookbehind to skip already replaced tokens
        text = re.sub(r'(?<!\[)' + r'[\w\.-]+@[\w\.-]+' + r'(?!\])', '[EMAIL]', text)
    
    return text

In [13]:
def anonymize_usernames(text, index, extracted_info_dict):
    """Find and replace usernames and URLs that contain the person's name.
    Looks for the first name written without spaces or with dashes, as commonly used in handles and URLs."""
    if not isinstance(text, str):
        return text
    
    if not indentified_name(index, extracted_info_dict):
        return text
    
    if not extracted_info_dict[index].get('name'):
        return text
    
    if index not in extracted_info_dict:
        extracted_info_dict[index] = {'username': []}

    name_str = extracted_info_dict[index].get('name', [])
    name = name_str[0].split()[0] if name_str and name_str[0].strip() else ''
    
    # Build patterns for the name without spaces and with dashes (e.g. johnsmith or john-smith)
    no_space = re.escape(re.sub(r'[\s-]', '', name))
    dashed = re.escape(name.lower().replace(" ", "-"))
    pattern = re.compile(f"{no_space}|{dashed}", re.IGNORECASE)
    
    # Match full tokens like URLs or handles that contain the username
    url_pattern = re.compile(r'(?<!\[)\S*(?:' + f"{no_space}|{dashed}" + r')\S*(?!\])', re.IGNORECASE)

    if re.search(pattern, text):
        extracted_info_dict[index]['username'] = re.sub(r'[\s-]', '', name).lower()
        text = re.sub(url_pattern, '[USERNAME]', text)

    return text

In [14]:
def identify_gender(index, extracted_info_dict, common_names):
    """Predict gender from the first name using a name-to-gender probability table.
    Uses probability ratios when a name appears for both genders."""

    if index not in extracted_info_dict or not extracted_info_dict[index].get('name'):
        extracted_info_dict.setdefault(index, {})['gender'] = 'unknown'
        return

    full_name = extracted_info_dict[index]['name'][0]
    first_name = full_name.split()[0].lower()

    matches = common_names[common_names['Name'].str.lower() == first_name]

    if matches.empty:
        extracted_info_dict[index]['gender'] = 'unknown'
        return

    male_prob = matches[matches['Gender'] == 'M']['Probability'].sum()
    female_prob = matches[matches['Gender'] == 'F']['Probability'].sum()

    if len(matches) == 1:
        extracted_info_dict[index]['gender'] = matches.iloc[0]['Gender']
    else:
        if (female_prob == 0 and male_prob == 0) or len(full_name) <= 2:
            extracted_info_dict[index]['gender'] = 'unknown'
        elif female_prob == 0:
            extracted_info_dict[index]['gender'] = 'M'
        elif male_prob == 0:
            extracted_info_dict[index]['gender'] = 'F'
        elif male_prob / female_prob >= 2:  # strongly male
            extracted_info_dict[index]['gender'] = 'M'
        elif female_prob / male_prob >= 2:  # strongly female
            extracted_info_dict[index]['gender'] = 'F'
        else:
            extracted_info_dict[index]['gender'] = 'unisex'

In [15]:
def anonymize_gender_words(text, index, extracted_info_dict):
    """Find gendered words that appear next to punctuation or newlines and replace them with [GENDER].
    Only targets words in those positions to avoid replacing common words mid-sentence."""
    if not isinstance(text, str):
        return text
    
    extracted_info_dict[index]['gender_words'] = []
    
    for gender, words in gender_words.items():
        for word in words:
            # Only match the word if it is preceded and followed by punctuation or a newline
            pattern = re.compile(
                r'(?<=[,.\-\n])' + re.escape(word) + r'(?=[,.\-\n]|$)', re.IGNORECASE)
            matches = re.findall(pattern, text)
            if matches:
                extracted_info_dict[index]['gender_words'].extend(matches)
                text = re.sub(pattern, '[GENDER]', text)
    
    return text

In [ ]:
def anonymize_gender_job_titles(text, index, job_titles_list, extracted_info_dict):
    """Replace gendered job titles with gender-neutral equivalents.
    For example, replaces fireman with firefighter."""
    if not isinstance(text, str):
        return text

    if 'neutralized_titles' not in extracted_info_dict[index]:
        extracted_info_dict[index]['neutralized_titles'] = []

    for gendered, neutral in job_titles_list.items():
        pattern = re.compile(r'\b' + re.escape(gendered) + r'\b', re.IGNORECASE)
        if re.search(pattern, text):
            extracted_info_dict[index]['neutralized_titles'].append(gendered)
            text = re.sub(pattern, neutral, text)

    return text

### Create anonymized file

In [17]:
resumes = pd.read_csv("raw_data/resumes.csv")  
gender_neutral_job_titles
extracted_info_dict = {}

# Anonymization
for idx, row in resumes.iterrows():
    text = row['Resume']
    text = anonymize_names(text, idx, extracted_info_dict)
    text = anonymize_emails(text, idx, extracted_info_dict)
    text = anonymize_usernames(text, idx, extracted_info_dict)
    identify_gender(idx, extracted_info_dict, common_names)
    text = anonymize_gender_words(text, idx, extracted_info_dict)
    text = anonymize_gender_job_titles(text, idx, gender_neutral_job_titles, extracted_info_dict)
    resumes.at[idx, 'anonymized_text'] = text


KeyboardInterrupt: 

In [ ]:
# Save anonymized texts without the target label
resumes[['anonymized_text']].to_csv("anonymized_files/anonymized_text.csv", index=False)

# Save anonymized texts with the target label
resumes[['anonymized_text']] \
    .assign(decision=data['decision'], 
            Job_Description=data['Job_Description'],
            gender=data['gender']) \
    .to_csv("anonymized_files/anonymized_text_target.csv", index=False)

# Save the full anonymized dataframe
resumes.to_csv("anonymized_files/anonymized_file.csv", index=False)

# Save the dictionary with all extracted information per resume
pd.DataFrame.from_dict(extracted_info_dict, orient='index').to_csv("anonymized_files/info_dictionary.csv")

NameError: name 'resumes' is not defined

In [19]:
# Extract gender from the dictionary and map it to a Series
gender_series = pd.Series({idx: info.get('gender', 'unknown') for idx, info in extracted_info_dict.items()})

# Save anonymized texts without the target label
resumes[['anonymized_text']].to_csv("anonymized_files/anonymized_text.csv", index=False)

# Save anonymized texts with the target label
resumes[['anonymized_text']] \
    .assign(decision=data['decision'], gender=gender_series, Job_Description=data['Job_Description']) \
    .to_csv("anonymized_files/anonymized_text_target.csv", index=False)

# Save the full anonymized dataframe
resumes.to_csv("anonymized_files/anonymized_file.csv", index=False)

# Save the dictionary with all extracted information per resume
pd.DataFrame.from_dict(extracted_info_dict, orient='index').to_csv("anonymized_files/info_dictionary.csv")